# EasyOCR synthetic-image benchmark

Two parts, one reader, same trial schedule.

**Part 1 — Baseline:** throughput, Type I/II errors, jersey label accuracy, and failure visualizations with OCR boxes. Runs with `preprocess="none"`.

**Part 2 — Preprocessing ablation:** repeats the same trials under `none / invert / clahe / clahe_invert / hsv_v` and compares speed + accuracy side by side.

Assets: `../images/base/` (10 images, no number) and `../images/with_number/` (≥1, expected jersey **5**). Run all cells top to bottom.


In [ ]:
import random
import re
import time
from pathlib import Path
from statistics import mean

import cv2
import easyocr
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image, ImageDraw
try:
    RESAMPLE = Image.Resampling.LANCZOS
except AttributeError:
    RESAMPLE = Image.LANCZOS  # Pillow < 9.1

EXPERIMENT_DIR = Path.cwd()
IMAGES_ROOT = EXPERIMENT_DIR.parent / "images"
BASE_DIR = IMAGES_ROOT / "base"
NUMBER_DIR = IMAGES_ROOT / "with_number"
EXT = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

N_SLOTS = 10
N_TRIALS = 50
P_HAS_NUMBER = 0.3
RANDOM_SEED = 42
WARMUP = True
EXPECTED_JERSEY = "5"

IMAGE_SCALE = 4
OCR_KWARGS = {
    "allowlist": "0123456789",
    "mag_ratio": 1.5,
    "contrast_ths": 0.05,
    "adjust_contrast": 0.7,
}
MIN_CONFIDENCE = 0.2
USE_GPU = None
MAX_DISPLAY = 30
SHOW_CORRECT_NUMBER_POOL = False

# Preprocessing variants for Part 2 (edit list to add/remove)
VARIANTS = ["none", "invert", "clahe", "clahe_invert", "hsv_v"]

REASON_COLORS = {
    "none": (220, 40, 40),
    "wrong": (220, 40, 40),
    "false_positive": (255, 140, 0),
    "ok": (40, 180, 80),
}


def list_images(folder: Path) -> list[Path]:
    return sorted(
        [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in EXT],
        key=lambda p: p.name.lower(),
    )


def load_rgb(path: Path) -> np.ndarray:
    im = Image.open(path).convert("RGB")
    if IMAGE_SCALE != 1:
        w, h = im.size
        im = im.resize((w * IMAGE_SCALE, h * IMAGE_SCALE), RESAMPLE)
    return np.array(im)


def preprocess_rgb(rgb: np.ndarray, mode: str) -> np.ndarray:
    """Return 3-channel RGB uint8 for EasyOCR."""
    if mode == "none":
        return rgb
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    if mode == "invert":
        return cv2.cvtColor(255 - gray, cv2.COLOR_GRAY2RGB)
    if mode == "clahe":
        lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)
        return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)
    if mode == "clahe_invert":
        out = preprocess_rgb(rgb, "clahe")
        return cv2.cvtColor(255 - cv2.cvtColor(out, cv2.COLOR_RGB2GRAY), cv2.COLOR_GRAY2RGB)
    if mode == "hsv_v":
        return cv2.cvtColor(cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)[:, :, 2], cv2.COLOR_GRAY2RGB)
    raise ValueError(f"Unknown preprocess mode: {mode}")


def parse_jersey_label(ocr_text: str) -> str:
    digits = re.sub(r"\D", "", ocr_text)
    return digits if digits else "(none)"


def parse_raw_detections(raw) -> list[dict]:
    out = []
    for item in raw:
        conf = float(item[2])
        text = str(item[1]).strip()
        if conf < MIN_CONFIDENCE or not text:
            continue
        out.append({"bbox": item[0], "text": text, "confidence": conf})
    return out


def read_frame(reader, rgb: np.ndarray) -> tuple[str, float, str, list[dict]]:
    t0 = time.perf_counter()
    raw = reader.readtext(rgb, detail=1, paragraph=False, **OCR_KWARGS)
    ms = (time.perf_counter() - t0) * 1000.0
    detections = parse_raw_detections(raw)
    text = " | ".join(d["text"] for d in detections) if detections else ""
    return text, ms, parse_jersey_label(text), detections


def label_error(expected: str, predicted: str, should_have_number: bool) -> str | None:
    if should_have_number:
        if predicted == "(none)":
            return "none"
        if predicted != expected:
            return "wrong"
        return None
    if predicted != "(none)":
        return "false_positive"
    return None


def build_trial(rng, bases: list[Path], numbers: list[Path]):
    paths = list(bases)
    injected = rng.random() < P_HAS_NUMBER
    slot = None
    if injected:
        slot = rng.randrange(N_SLOTS)
        paths[slot] = rng.choice(numbers)
    return paths, injected, slot


def draw_annotated(path: Path, detections: list[dict], reason: str | None) -> Image.Image:
    im = Image.fromarray(load_rgb(path))
    draw = ImageDraw.Draw(im)
    color = REASON_COLORS.get(reason or "", (255, 255, 0))
    for det in detections:
        pts = [(int(p[0]), int(p[1])) for p in det["bbox"]]
        draw.polygon(pts, outline=color, width=3)
        x = min(p[0] for p in pts)
        y = min(p[1] for p in pts) - 14
        draw.text((x, max(0, y)), f"{det['text']} ({det['confidence']:.2f})", fill=color)
    return im


def append_failure(wrong_labels, *, path, trial, slot, expected, predicted, ocr_text, reason, detections):
    wrong_labels.append({
        "image": path.name,
        "path": path,
        "trial": trial,
        "slot": slot,
        "expected": expected,
        "predicted": predicted,
        "ocr_text": ocr_text or "(none)",
        "reason": reason,
        "detections": detections,
    })


In [ ]:
bases = list_images(BASE_DIR)
numbers = list_images(NUMBER_DIR)
if len(bases) != N_SLOTS:
    raise SystemExit(f"Need {N_SLOTS} images in {BASE_DIR}, found {len(bases)}")
if not numbers:
    raise SystemExit(f"Need at least 1 image in {NUMBER_DIR}")

gpu = USE_GPU
if gpu is None:
    try:
        import torch
        gpu = torch.cuda.is_available()
    except ImportError:
        gpu = False

print("Loading EasyOCR...")
t0 = time.perf_counter()
reader = easyocr.Reader(["en"], gpu=gpu, verbose=False)
init_s = time.perf_counter() - t0
print(f"Reader ready in {init_s:.1f}s (gpu={gpu})")

# Shared trial schedule — both parts use the same seed so results are comparable
rng_shared = random.Random(RANDOM_SEED)
shared_trials = [build_trial(rng_shared, bases, numbers) for _ in range(N_TRIALS)]

if WARMUP:
    read_frame(reader, load_rgb(bases[0]))

number_names = {p.name for p in numbers}
base_names = {p.name for p in bases}
print(f"Images: {len(bases)} base, {len(numbers)} with_number")
print(f"Trials: {N_TRIALS}, seed={RANDOM_SEED}")


## Part 1 — Baseline

Throughput, Type I/II errors, jersey label accuracy, and failure visualizations. Runs with `preprocess="none"`.


In [ ]:
wrong_labels: list[dict] = []
preflight_ok: list[dict] = []

print(f"Preflight with_number/ (expected jersey = {EXPECTED_JERSEY}):")
for path in numbers:
    text, _, predicted, detections = read_frame(reader, load_rgb(path))
    err = label_error(EXPECTED_JERSEY, predicted, should_have_number=True)
    status = "OK" if err is None else err.upper()
    print(f"  [{status:16s}]  {path.name:40s}  got {predicted}")
    if err is None:
        preflight_ok.append({"path": path, "predicted": predicted, "detections": detections})
    else:
        append_failure(
            wrong_labels, path=path, trial="preflight", slot="-",
            expected=EXPECTED_JERSEY, predicted=predicted,
            ocr_text=text, reason=err, detections=detections,
        )

frame_ms: list[float] = []
batch_ms: list[float] = []
tp = fp = fn = tn = 0
trials_with_number = 0
trials_without_number = 0

for trial, (paths, injected, inject_slot) in enumerate(shared_trials):
    trials_with_number += injected
    trials_without_number += not injected
    t_batch = time.perf_counter()
    for slot, path in enumerate(paths):
        text, ms, predicted, detections = read_frame(reader, load_rgb(path))
        frame_ms.append(ms)
        is_number_slot = injected and slot == inject_slot
        err = label_error(EXPECTED_JERSEY, predicted, should_have_number=is_number_slot)
        if is_number_slot and predicted == EXPECTED_JERSEY:
            tp += 1
        elif is_number_slot:
            fn += 1
        elif predicted != "(none)":
            fp += 1
        else:
            tn += 1
        if err:
            append_failure(
                wrong_labels, path=path, trial=trial, slot=slot,
                expected=EXPECTED_JERSEY if is_number_slot else "(none)",
                predicted=predicted, ocr_text=text, reason=err, detections=detections,
            )
    batch_ms.append((time.perf_counter() - t_batch) * 1000.0)

print(f"\nDone: {N_TRIALS} trials, {len(frame_ms)} frames, {len(wrong_labels)} failures logged")


In [ ]:
n_number_slots = tp + fn
n_plain_slots = fp + tn
type_i_pct = 100.0 * fp / n_plain_slots if n_plain_slots else 0.0
type_ii_pct = 100.0 * fn / n_number_slots if n_number_slots else 0.0
label_miss = sum(1 for w in wrong_labels if w["reason"] == "none" and w["expected"] == EXPECTED_JERSEY)
label_bad = sum(1 for w in wrong_labels if w["reason"] == "wrong" and w["expected"] == EXPECTED_JERSEY)

print("=" * 50)
print("EASYOCR BASELINE (preprocess=none)")
print("=" * 50)
print(f"trials:              {N_TRIALS}")
print(f"expected jersey:     {EXPECTED_JERSEY}")
print(f"trials w/ number:    {trials_with_number}")
print(f"trials w/o number:   {trials_without_number}")
print(f"image_scale:         {IMAGE_SCALE}  gpu: {gpu}")
print()
print("SPEED")
print(f"  reader init:       {init_s:.2f} s")
print(f"  mean per frame:    {mean(frame_ms):.1f} ms")
print(f"  mean per batch:    {mean(batch_ms):.1f} ms  ({mean(batch_ms)/1000:.2f} s)")
print(f"  batch fps:         {1000.0 * N_SLOTS / mean(batch_ms):.2f}")
print()
print("ERRORS")
print(f"  type I  false positive:  {fp:4d}  ({type_i_pct:.1f}% of non-number frames)")
print(f"  type II false negative:  {fn:4d}  ({type_ii_pct:.1f}% of number slots)")
print(f"  true positive:           {tp:4d}")
print(f"  true negative:           {tn:4d}")
print()
print(f"LABEL (expected {EXPECTED_JERSEY} on number slots)")
if n_number_slots:
    print(f"  correct:                 {tp:4d}  ({100 * tp / n_number_slots:.1f}%)")
print(f"  wrong digit (not {EXPECTED_JERSEY}):   {label_bad:4d}")
print(f"  not labelled (none):     {label_miss:4d}")
print()
print("WRONG LABELS")
if not wrong_labels:
    print("  (none)")
else:
    for w in wrong_labels:
        print(
            f"  {w['image']}  trial={w['trial']} slot={w['slot']}"
            f"  expected={w['expected']} got={w['predicted']}"
            f"  [{w['reason']}]  raw=\"{w['ocr_text']}\""
        )
print()
print("NUMBER POOL — mislabelled at least once")
any_number = False
for name in sorted(number_names):
    fails = [w for w in wrong_labels if w["image"] == name]
    if not fails:
        continue
    any_number = True
    preds = sorted({w["predicted"] for w in fails})
    reasons = sorted({w["reason"] for w in fails})
    print(f"  {name}  got={','.join(preds)}  reasons={','.join(reasons)}")
if not any_number:
    print("  (none)")
print()
print("BASE — false positives at least once")
any_base = False
for name in sorted(base_names):
    fails = [w for w in wrong_labels if w["image"] == name and w["reason"] == "false_positive"]
    if not fails:
        continue
    any_base = True
    preds = sorted({w["predicted"] for w in fails})
    print(f"  {name}  got={','.join(preds)}")
if not any_base:
    print("  (none)")
print("=" * 50)


In [ ]:
seen = set()
failures_unique: list[dict] = []
for w in wrong_labels:
    key = (w["image"], str(w["trial"]), str(w["slot"]))
    if key in seen:
        continue
    seen.add(key)
    failures_unique.append(w)

to_show: list[dict] = list(failures_unique)
if SHOW_CORRECT_NUMBER_POOL:
    for ok in preflight_ok:
        to_show.append({
            "path": ok["path"], "image": ok["path"].name, "trial": "preflight",
            "slot": "-", "expected": EXPECTED_JERSEY, "predicted": ok["predicted"],
            "reason": "ok", "detections": ok["detections"],
        })

truncated = len(to_show) > MAX_DISPLAY
if truncated:
    print(f"Showing first {MAX_DISPLAY} of {len(to_show)} images (set MAX_DISPLAY to raise cap).\n")
    to_show = to_show[:MAX_DISPLAY]
elif not to_show:
    print("No failures to visualize.")
else:
    print(f"Showing {len(to_show)} failure image(s) with OCR boxes.\n")

for w in to_show:
    im = draw_annotated(w["path"], w["detections"], w["reason"])
    print(
        f"{w['image']}  trial={w['trial']} slot={w['slot']}  "
        f"expected={w['expected']} got={w['predicted']}  [{w['reason']}]"
    )
    display(im)


## Part 2 — Preprocessing ablation

Runs the same `shared_trials` schedule under each variant in `VARIANTS` and compares speed + accuracy in one table. Speed is measured per-variant so any preprocess overhead is captured alongside OCR time.


In [ ]:
ablation_results: list[dict] = []

for variant in VARIANTS:
    print(f"\n--- Running variant: {variant} ---")
    frame_ms_v: list[float] = []
    batch_ms_v: list[float] = []
    tp_v = fp_v = fn_v = tn_v = 0
    preflight_ok_v = preflight_miss_v = preflight_wrong_v = 0

    for path in numbers:
        rgb = preprocess_rgb(load_rgb(path), variant)
        _, _, predicted, _ = read_frame(reader, rgb)
        err = label_error(EXPECTED_JERSEY, predicted, should_have_number=True)
        if err is None:
            preflight_ok_v += 1
        elif err == "none":
            preflight_miss_v += 1
        else:
            preflight_wrong_v += 1

    for paths, injected, inject_slot in shared_trials:
        t_batch = time.perf_counter()
        for slot, path in enumerate(paths):
            t0 = time.perf_counter()
            rgb = preprocess_rgb(load_rgb(path), variant)
            _, ms, predicted, _ = read_frame(reader, rgb)
            frame_ms_v.append((time.perf_counter() - t0) * 1000.0)  # includes preprocess
            is_number_slot = injected and slot == inject_slot
            if is_number_slot and predicted == EXPECTED_JERSEY:
                tp_v += 1
            elif is_number_slot:
                fn_v += 1
            elif predicted != "(none)":
                fp_v += 1
            else:
                tn_v += 1
        batch_ms_v.append((time.perf_counter() - t_batch) * 1000.0)

    n_number_v = tp_v + fn_v
    n_plain_v = fp_v + tn_v
    ablation_results.append({
        "preprocess": variant,
        "mean_frame_ms": round(mean(frame_ms_v), 1),
        "mean_batch_ms": round(mean(batch_ms_v), 1),
        "batch_fps": round(1000.0 * N_SLOTS / mean(batch_ms_v), 2),
        "type_i_count": fp_v,
        "type_i_pct": round(100.0 * fp_v / n_plain_v, 1) if n_plain_v else 0.0,
        "type_ii_count": fn_v,
        "type_ii_pct": round(100.0 * fn_v / n_number_v, 1) if n_number_v else 0.0,
        "true_positive": tp_v,
        "true_negative": tn_v,
        "number_slot_accuracy_pct": round(100.0 * tp_v / n_number_v, 1) if n_number_v else 0.0,
        "preflight_ok": preflight_ok_v,
        "preflight_total": len(numbers),
        "preflight_ok_pct": round(100.0 * preflight_ok_v / len(numbers), 1),
        "preflight_miss": preflight_miss_v,
        "preflight_wrong": preflight_wrong_v,
    })
    print(f"  preflight {preflight_ok_v}/{len(numbers)} ok | mean frame {mean(frame_ms_v):.1f} ms")

print("\nAll variants finished.")


In [ ]:
ab = pd.DataFrame(ablation_results).set_index("preprocess")

ab_display = ab.copy()
ab_display["rank_speed"] = ab["mean_frame_ms"].rank(method="min")
ab_display["rank_accuracy"] = ab["number_slot_accuracy_pct"].rank(ascending=False, method="min")
ab_display["rank_preflight"] = ab["preflight_ok_pct"].rank(ascending=False, method="min")
ab_display["rank_type_i"] = ab["type_i_pct"].rank(method="min")
ab_display["rank_type_ii"] = ab["type_ii_pct"].rank(method="min")

print("=" * 72)
print("PREPROCESSING ABLATION — side-by-side (same trials, seed=%d, n=%d)" % (RANDOM_SEED, N_TRIALS))
print("=" * 72)
display(ab.round(1))

print("\nRank columns (1 = best among variants):")
display(ab_display[["rank_speed", "rank_preflight", "rank_accuracy", "rank_type_i", "rank_type_ii"]].astype(int))

best = {
    "fastest_frame_ms": ab["mean_frame_ms"].idxmin(),
    "best_preflight": ab["preflight_ok_pct"].idxmax(),
    "best_number_slots": ab["number_slot_accuracy_pct"].idxmax(),
    "lowest_type_i": ab["type_i_pct"].idxmin(),
    "lowest_type_ii": ab["type_ii_pct"].idxmin(),
}
print("\nBest variant per metric:")
for k, v in best.items():
    print(f"  {k}: {v}")

if "none" in ab.index:
    delta = ab.sub(ab.loc["none"])
    delta.columns = [f"delta_{c}" for c in delta.columns]
    print("\nDelta vs preprocess=none (negative ms / errors = improvement):")
    display(delta.drop(index="none", errors="ignore").round(1))


In [ ]:
# Per-image × variant preflight grid
rows = []
for path in numbers:
    row = {"image": path.name}
    for variant in VARIANTS:
        rgb = preprocess_rgb(load_rgb(path), variant)
        _, _, pred, _ = read_frame(reader, rgb)
        row[variant] = pred
    rows.append(row)

preflight_grid = pd.DataFrame(rows).set_index("image")
print("Preflight predicted label per image (expected %s):" % EXPECTED_JERSEY)
display(preflight_grid)

if len(preflight_grid.columns):
    miss_count = (preflight_grid != EXPECTED_JERSEY).sum()
    print("\nMiss/wrong count per variant:")
    display(miss_count.to_frame("not_correct").sort_values("not_correct"))
